# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/peddikotlahimani/Flyrank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: pages that show up in search a lot (high impressions) and rank near the top (low position number) are likely to get more clicks in the future. So I score every page as either 'good' or 'not good' based on these two numbers.

no_impressions : Page had 0 impressions — nothing to judge, no signal at all
good_visibility :	Page has decent impressions AND ranks in the top 10 — likely to get clicks
poor_visibility :	Page is missing one or both of those — impressions too low, or position too weak

In [ ]:
from google.colab import userdata
import duckdb

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

my_token = userdata.get("token_dataaccess")
con.sql(f"""
CREATE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{my_token}'
);
""")

raw_data = con.sql("""
    SELECT content_hash_id, client_hash_id, report_date,
           gsc_clicks, gsc_impressions, gsc_avg_position,
           ga4_engaged_sessions, ga4_sessions, ga4_data_available
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()

raw_data = raw_data[raw_data["ga4_data_available"] == True]
raw_data = raw_data[raw_data["report_date"] < "2026-03-16"]
raw_data = raw_data.drop(columns=["report_date"])

feature_vector = raw_data.groupby(["content_hash_id", "client_hash_id"]).agg({
    "gsc_clicks": "sum",
    "gsc_impressions": "sum",
    "gsc_avg_position": "mean",
    "ga4_engaged_sessions": "sum",
    "ga4_sessions": "sum"
}).reset_index()

feature_vector["ctr_1h"] = feature_vector["gsc_clicks"] / feature_vector["gsc_impressions"]
feature_vector["engagement_rate_1h"] = feature_vector["ga4_engaged_sessions"] / feature_vector["ga4_sessions"]
feature_vector["had_impressions"] = feature_vector["gsc_impressions"] > 0
feature_vector["ctr_1h"] = feature_vector["ctr_1h"].fillna(0)
feature_vector["gsc_avg_position"] = feature_vector["gsc_avg_position"].fillna(100)

print("Feature vector rebuilt. Rows:", len(feature_vector))

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.